In [1]:
from ollama import chat
from ollama import ChatResponse
from ollama import Client
import pandas as pd
import json
import time
import random
import numpy as np
import os
from tqdm import tqdm
import gc
import re
import warnings
warnings.filterwarnings("ignore")

### Setup RAG

- Define output schema

In [ ]:
from typing import List
from pydantic import BaseModel


class prediction(BaseModel):
    T: str
    N: str
    stage: str
    reason: str

json_schema = prediction.model_json_schema()
json_schema


- create document converter

In [ ]:
# Create document store and preprocessing pipeline with metadata enrichment
from haystack import Pipeline, component
from haystack.components.writers import DocumentWriter
from haystack.components.converters import MarkdownToDocument, PyPDFToDocument, TextFileToDocument
from haystack.components.preprocessors import DocumentSplitter, DocumentCleaner
from haystack.components.routers import FileTypeRouter
from haystack.components.joiners import DocumentJoiner
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document

from typing import List, Dict, Any
from pathlib import Path
from datetime import datetime
import mimetypes

# Embedding model and document store
# Using BiomedNLP-PubMedBERT for better medical terminology understanding
embedding_model = "all-mpnet-base-v2"  # High-quality model for scientific/medical papers
document_store = InMemoryDocumentStore()

# File routing and converters
file_type_router = FileTypeRouter(mime_types=["text/plain", "application/pdf", "text/markdown"])
text_file_converter = TextFileToDocument()
markdown_converter = MarkdownToDocument()
pdf_converter = PyPDFToDocument()

# Preprocessing components
document_joiner = DocumentJoiner()
document_cleaner = DocumentCleaner()
document_splitter = DocumentSplitter(split_by="word", split_length=150, split_overlap=30)

document_embedder = SentenceTransformersDocumentEmbedder(model=embedding_model)
document_writer = DocumentWriter(document_store)

# Custom component to enrich metadata
@component
class MetadataEnricher:
    @component.output_types(documents=List[Document])
    def run(self, documents: List[Document]) -> Dict[str, Any]:
        for doc in documents:
            meta = getattr(doc, "meta", None)
            if meta is None:
                doc.meta = {}
                meta = doc.meta

            # Source information
            file_path = meta.get("file_path")
            if file_path:
                meta["source_file"] = str(file_path)
                meta["file_name"] = Path(file_path).name
                #cancer type=file name onyly
                meta['cancer_type'] = Path(file_path).name.split('.')[0]
                meta["source_dir"] = str(Path(file_path).parent)
                mime = mimetypes.guess_type(str(file_path))[0]
                if mime:
                    meta["mime_type"] = mime

            # Processing info
            meta["processed_at"] = datetime.now().isoformat()
            meta["ingestion_pipeline"] = "preprocess_v1"

            # Content info
            if getattr(doc, "content", None):
                meta["content_length"] = len(doc.content)
                if "content_preview" not in meta:
                    meta["content_preview"] = doc.content[:120]
        return {"documents": documents}

metadata_enricher = MetadataEnricher()

# Build preprocessing pipeline with metadata enrichment
preprocessing_pipeline = Pipeline()
preprocessing_pipeline.add_component(instance=file_type_router, name="file_type_router")
preprocessing_pipeline.add_component(instance=text_file_converter, name="text_file_converter")
preprocessing_pipeline.add_component(instance=markdown_converter, name="markdown_converter")
preprocessing_pipeline.add_component(instance=pdf_converter, name="pypdf_converter")
preprocessing_pipeline.add_component(instance=document_joiner, name="document_joiner")
preprocessing_pipeline.add_component(instance=document_cleaner, name="document_cleaner")
preprocessing_pipeline.add_component(instance=document_splitter, name="document_splitter")
preprocessing_pipeline.add_component(instance=metadata_enricher, name="metadata_enricher")
preprocessing_pipeline.add_component(instance=document_embedder, name="document_embedder")
preprocessing_pipeline.add_component(instance=document_writer, name="document_writer")

# Connect the components
preprocessing_pipeline.connect("file_type_router.text/plain", "text_file_converter.sources")
preprocessing_pipeline.connect("file_type_router.application/pdf", "pypdf_converter.sources")
preprocessing_pipeline.connect("file_type_router.text/markdown", "markdown_converter.sources")
preprocessing_pipeline.connect("text_file_converter", "document_joiner")
preprocessing_pipeline.connect("pypdf_converter", "document_joiner")
preprocessing_pipeline.connect("markdown_converter", "document_joiner")
preprocessing_pipeline.connect("document_joiner", "document_cleaner")
preprocessing_pipeline.connect("document_cleaner", "document_splitter")
preprocessing_pipeline.connect("document_splitter", "metadata_enricher")
preprocessing_pipeline.connect("metadata_enricher", "document_embedder")
preprocessing_pipeline.connect("document_embedder", "document_writer")

print("Preprocessing pipeline with metadata enrichment is ready.")

In [ ]:
# Ingest all files from the folder into the document store
# Collect files (recursively) under the folder
folder = r"E:\head_neck_staging\knowledge_document"

sources = [p for p in Path(folder).rglob("*") if p.is_file()]
print(f"Found {len(sources)} files to ingest from: {folder}")

# Run the pipeline
if sources:
    preprocessing_pipeline.run({"file_type_router": {"sources": sources}})
    print("Ingestion completed.")
else:
    print("No files found to ingest.")

In [ ]:
from docx import Document as DocxDocument

def docx_to_plain_documents(paths):
    docs=[]
    for p in paths:
        try:
            d=DocxDocument(str(p))
            full_text=[]
            for para in d.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
            content="\n".join(full_text)
            docs.append(Document(content=content, meta={"file_path": str(p), "original_ext": ".docx"}))
        except Exception as e:
            print(f"Failed to read {p}: {e}")
    return docs

# Use manual conversion then continue with splitter->enricher->embedder->writer
manual_docx_sources=[p for p in Path(folder).rglob("*.docx")]
manual_docs=docx_to_plain_documents(manual_docx_sources)
print(f"Manually converted {len(manual_docs)} docx files.")

# Run remaining pipeline components manually since we bypassed router/converters
cleaned = document_cleaner.run(documents=manual_docs)["documents"]
split_docs = document_splitter.run(documents=cleaned)["documents"]
enriched = metadata_enricher.run(documents=split_docs)["documents"]
embedded = document_embedder.run(documents=enriched)["documents"]
_ = document_writer.run(documents=embedded)
print(f"Document store now has {len(document_store.filter_documents())} documents after manual DOCX ingestion.")

In [ ]:
# Verify stored documents and metadata
all_docs = document_store.filter_documents()
print(f"Total documents in store: {len(all_docs)}")

for i, doc in enumerate(all_docs[:3]):
    print(f"\n--- Document {i+1} ---")
    print(f"Content preview: {doc.content[:120] if doc.content else ''}")
    print(f"Metadata: {doc.meta}")
    print(f"Embedding present: {doc.embedding is not None}")

### Setup LLM

- create LLM client

In [ ]:
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack.dataclasses import ChatMessage

model_name = "gpt-oss:20b"
seed_list=[15,2655,454233,9672678,132323989]
seed_num = seed_list[4]
print(f"Using seed number: {seed_num}")
generator = OllamaChatGenerator(model=model_name, timeout=200, url='http://localhost:11434',
                             generation_kwargs={
                              "top_k": 5,
                              "temperature": 0.1,
                              "seed": seed_num,
                              "format": json_schema,
                              #"Content-Type":'application/json', 
                              "timeout": 200
                              })


In [ ]:
from haystack.dataclasses import ChatMessage
def create_rag_template():
    system_message = ChatMessage.from_system(
        """
        You are a medical AI assistant with expertise in head neck cancer staging and management. Your task is to classify cancer stage that require the pathologic repoart and/or radiology report.
        T staging is based on the size and extent of the primary tumor. N staging is based on the involvement of regional lymph nodes.
        Overall stage grouping combines T and N classifications to determine the overall stage of the cancer.
        Follow this systematic approach:
        Step 1. Determine primary cancer site
        Step 2. Primary tumor assessment from pathologic report (if available)(1st priority) and radiologic investigation  (if available).
        Step 3. T stage assessment according to AJCC staging criteria depend on primary site and findings from step 2.
        Step 4. Neck lymph node assessment from pathologic report (if available)(1st priority) and radiologic investigation  (if available).
        Step 5. N stage assessment according to AJCC staging criteria depend on primary site and findings from step 4.
        Step 6. AJCC staging classification (8th edition) for head neck cancer.
        *Note :  -DOI is depth of invasion and not tumor thickness.
                 -ENE is extranodal extension.
                 -The patholoic lymph node in radiology report is size more than 1 cm and show necrosis or irregular margin to consider as pathologic lymph node. 
                 - Pathologic node location is affeected the AJCC N staging base on cancer type.
        If you are not sure about the stage, try your best to answer. Answer in JSON format as example below :
        * Note:and do not use (```json) in the response
        {{schema}}
        This is an example response:
        {"T": "T2",
        "N": "N1",
        "stage": "Stage III",
        "reason": "Based on the primary tumor size of 3 cm and involvement of a single ipsilateral lymph node less than 3 cm without ENE, the T2 and N1 classifications lead to an overall Stage III classification according to AJCC 8th edition criteria."
         """
    )
    
    user_message = ChatMessage.from_user(
        """
        "Context information:\n"
        "{% for document in documents %}"
        "{{ document.content }}\n"
        "{% endfor %}"
        "\n"
        From the provided patient information . please classify patient stage according to AJCC staging. Answer T, N and overall stage (all case are M0).
        The AJCC staging document of each cancer is specified in the document meta 'cancer_type'. 
         \n
         This is the document(Input) that you have to classify stage
         \n
      {{question}}
   """
    )
    return [system_message, user_message]

template = create_rag_template()
template

In [ ]:
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
#from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

# Build RAG pipeline with optimized retrieval
pipe = Pipeline()
pipe.add_component("embedder", SentenceTransformersTextEmbedder(model=embedding_model))
# Increase top_k to retrieve more documents for better context
pipe.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store, top_k=5))
pipe.add_component("chat_prompt_builder", ChatPromptBuilder(template=template))
pipe.add_component(
    "llm",
    generator,
)

pipe.connect("embedder.embedding", "retriever.query_embedding")
pipe.connect("retriever.documents", "chat_prompt_builder.documents")
pipe.connect("chat_prompt_builder.prompt", "llm.messages")

- Load data

In [ ]:
#load data
df=pd.read_excel(path)
print(f"Initial data shape: {df.shape}")
df

### Run 1 sample

In [ ]:
data=df.iloc[0]
data

In [ ]:
#create question from ecog, ChildPugh, DOctorNote using f
question = f"""This patient had cancer of {data['diagnosis']} . He/She had radiology report as {data['Radiology_result']} and pathologic result was {data['Pathology_result']} """
question

In [ ]:
response=pipe.run({"embedder": {"text": question}, "chat_prompt_builder": {"question": question,"schema": json_schema}})
response

In [36]:
def extract_llm_response(response):
    c=response['llm']['replies'][0].text
    match = re.search(r'\{.*\}', c, re.DOTALL)
    if match:
     json_str = match.group(0)
     try:
        data = json.loads(json_str)
        return data
     except json.JSONDecodeError as e:
        print(f"Failed to parse: {e}")

In [ ]:
json_resp=extract_llm_response(response)
json_resp

### run 1 seed

In [ ]:
df_sample=df.copy()
df_sample['seed'] = seed_num
df_sample['response']=''
df_sample['T_prediction'] = ''
df_sample['N_prediction']=''
df_sample['Stage_prediction']=''
df_sample['reason']=''  
df_sample['Time']=''
success_count=0
failure_count=0
df_sample

In [39]:
def extract_llm_response(response):
    c=response['llm']['replies'][0].text
    match = re.search(r'\{.*\}', c, re.DOTALL)
    if match:
     json_str = match.group(0)
     try:
        data = json.loads(json_str)
        return data
     except json.JSONDecodeError as e:
        print(f"Failed to parse: {e}")

In [40]:
def compute_response_time(response):
 meta = response['llm']['replies'][0].meta
 time=meta['total_duration']/1_000_000_000
 return    time

In [ ]:
#model_name = "qwen3:32b" 
#df_sample=df.copy()
#start_time = time.time()
success_count=0
failure_count=0
for i in tqdm(range(len(df_sample))):
    
    dx=df_sample['diagnosis'].iloc[i]
    radio=df_sample['Radiology_result'].iloc[i]
    patho=df_sample['Pathology_result'].iloc[i]


    question = f"""This patient had cancer of {dx} . He/She had radiology report as {radio} and pathologic result was {patho} """
    try:
     response=pipe.run({"embedder": {"text": question}, "chat_prompt_builder": {"question": question,"schema": json_schema}})
     time_taken=compute_response_time(response)
     try:
      result_json=extract_llm_response(response)
      df_sample['T_prediction'].iloc[i] = result_json['T']
      df_sample['N_prediction'].iloc[i] = result_json['N']
      df_sample['Stage_prediction'].iloc[i] = result_json['stage']
      df_sample['reason'].iloc[i] = result_json['reason'] 
      df_sample['Time'].iloc[i]=time_taken
      print("✓ LLM extraction successful")
      success_count+=1
      print(f"T: {result_json['T']}, N: {result_json['N']}, Stage: {result_json['stage']}")
     
     except:
      df_sample['response'].iloc[i] = response['llm']['replies'][0].text
      df_sample['Time'].iloc[i]=time_taken
      print("✗ LLM extraction failed")
      failure_count+=1
    except:
     df_sample['response'].iloc[i] = "RAG pipeline failed"
     failure_count+=1
     print("✗ RAG pipeline failed")
print(f"Total successful extractions: {success_count}")
print(f"Total failed extractions: {failure_count}")

### Check result

In [47]:
def llm_extract_staging(raw_response: str, generator_instance) -> dict:
    """
    Use LLM to extract T, N, and stage from raw LLM response.
    
    Args:
        raw_response: Raw text response from the staging LLM
        generator_instance: Your OllamaChatGenerator instance
    
    Returns:
        dict: Extracted {T, N, stage, reason} in JSON format
    """
    from haystack.dataclasses import ChatMessage
    import json
    import re
    
    extraction_prompt = f"""You are a JSON extraction assistant. Extract the cancer staging information from the following text and return ONLY a valid JSON object with these exact keys: T, N, stage, reason.

Text to extract from:
{raw_response}

Return your answer as a JSON object with this exact format:
{{
    "T": "extracted T stage value",
    "N": "extracted N stage value", 
    "stage": "extracted overall stage value",
    "reason": "extracted reasoning or explanation"
}}

Important rules:
- Extract ONLY the stage values (e.g., "T4a", "N2a", "Stage IVa" or "IVa")
- Remove any prefixes like "Stage:" or "T:" from the values
- Remove (M0) from stage if present
- If a field is not found, use empty string ""
- Return ONLY valid JSON, no other text or markdown formatting"""

    messages = [ChatMessage.from_user(extraction_prompt)]
    
    try:
        extraction_response = generator_instance.run(messages=messages)
        extracted_text = extraction_response['replies'][0].text
        
        # Remove code fences if present
        extracted_text = re.sub(r"```(?:json)?\s*([\s\S]*?)```", lambda m: m.group(1), extracted_text).strip()
        
        # Find JSON object
        json_match = re.search(r'\{[\s\S]*?\}', extracted_text)
        if json_match:
            result = json.loads(json_match.group(0))
            
            # Validate and normalize keys
            validated = {
                'T': str(result.get('T', '')),
                'N': str(result.get('N', '')),
                'stage': str(result.get('stage', '')),
                'reason': str(result.get('reason', ''))
            }
            
            return validated
        else:
            print("⚠ LLM extraction did not return valid JSON")
            return None
            
    except Exception as e:
        print(f"⚠ LLM extraction failed: {e}")
        return None